# 4교시 · 데이터 탐색과 정리
### — 요약한 숫자를 의심해 보기

앞 시간에 계속 **합계와 평균**을 냈습니다.
이번 시간에는 이렇게 묻습니다. **그 평균, 믿어도 됩니까?**

**이 시간이 끝나면 할 수 있는 것**

1. 평균과 중앙값이 왜 다른지 설명할 수 있다
2. 분포를 그려 보고 데이터의 모양을 확인할 수 있다
3. 표준편차가 무엇을 말해 주는지 안다
4. **이상치를 찾고, 지울지 남길지 스스로 판단할 수 있다**

In [ ]:
import pandas as pd

BASE = 'https://raw.githubusercontent.com/JasonWhiteLee/ak-data-analysis-basics/main/'

orders = pd.read_csv(BASE + 'superstore_orders.csv', parse_dates=['Order Date', 'Ship Date'])
orders = orders.drop_duplicates()

print(orders.shape)

## 그래프에 한글이 나오게 하기

Colab 은 기본 상태에서 **한글을 네모(□□□)로 그립니다.** 폰트가 없기 때문입니다.
아래 두 셀을 먼저 실행하세요. 오늘 이후 그래프를 그릴 때마다 필요합니다.

In [ ]:
!apt-get -qq install fonts-nanum > /dev/null

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

fm.fontManager.addfont('/usr/share/fonts/truetype/nanum/NanumGothic.ttf')
plt.rc('font', family='NanumGothic')
plt.rc('axes', unicode_minus=False)      # 마이너스 기호도 깨지므로 함께 설정

plt.plot([1, 2, 3], [1, 4, 2])
plt.title('한글이 보이면 성공입니다')
plt.show()

---
# 4-1. 평균 하나로 보고하면 생기는 일

주문 한 건당 매출이 얼마인지 보겠습니다.

In [ ]:
orders['Sales'].mean().round(1)

**평균 주문 금액은 239.2 입니다.**

이 숫자를 보고 이런 생각이 듭니다 — "우리 고객은 보통 240 정도 쓰는구나."

**확인해 봅시다.** 정말 240 근처에 사람들이 몰려 있을까요?

In [ ]:
print('평균 :', orders['Sales'].mean().round(1))
print('중앙값:', orders['Sales'].____.round(1))

## 평균 239.2, 중앙값 53.7

**4배 넘게 차이납니다.**

- **평균** — 다 더해서 개수로 나눈 값
- **중앙값** — 크기순으로 줄 세웠을 때 **한가운데** 있는 값

중앙값이 53.7 이라는 건, **주문의 절반이 53.7 이하**라는 뜻입니다.
평균 239.2 는 대부분의 주문과 거리가 멉니다.

실제로 세어 봅시다.

In [ ]:
평균보다큼 = (orders['Sales'] > orders['Sales'].mean())

print('평균보다 큰 주문: {:.1f}%'.format(평균보다큼.mean() * 100))
print('평균보다 작은 주문: {:.1f}%'.format((~평균보다큼).mean() * 100))

## 주문의 77%가 평균 이하입니다

"평균 주문금액 239" 라고 보고하면, 듣는 사람은 보통 그 정도 주문이 많다고 이해합니다.
**실제로는 열에 여덟이 그보다 적게 씁니다.**

왜 이렇게 됐을까요? 큰 주문 몇 개가 평균을 끌어올렸기 때문입니다.

In [ ]:
상위1퍼센트 = orders['Sales'].nlargest(int(len(orders) * 0.01))

print('상위 1% 주문이 전체 매출에서 차지하는 비중: {:.1f}%'.format(
      상위1퍼센트.sum() / orders['Sales'].sum() * 100))

**상위 1% 주문(약 100건)이 전체 매출의 23%를 차지합니다.**

> ### 평균이 버리는 것
> 평균은 여러 값을 **하나로 줄인 숫자**입니다. 줄이는 과정에서 정보가 사라집니다.
> 사라진 정보는 **"값들이 어떻게 흩어져 있는가"** 입니다.
>
> 이 데이터처럼 한쪽에 몰려 있고 몇 개가 아주 큰 경우,
> 평균은 **어느 쪽도 대표하지 못합니다.**
> 다수의 작은 주문도 아니고, 소수의 큰 주문도 아닌 값이 됩니다.

## 그럼 무엇을 보고해야 할까요

| 상황 | 어울리는 값 |
|---|---|
| 값들이 고르게 퍼져 있다 | **평균** |
| 한쪽으로 치우쳐 있다 (소득, 매출, 방문수 등) | **중앙값** |
| 큰 값 자체가 중요하다 (총매출 예측) | **평균 또는 합계** |

**정답은 없습니다.** 다만 **하나만 보고하면 안 됩니다.**
평균과 중앙값이 크게 다르다면, 그 사실 자체가 알려야 할 정보입니다.

---
# 4-2. 분포 — 데이터의 모양을 직접 본다

숫자 두 개(평균·중앙값)로 짐작하는 대신, **그림으로 보면 한 번에 알 수 있습니다.**

**히스토그램**은 값의 구간별로 몇 개가 있는지 막대로 그린 그림입니다.

In [ ]:
orders['Sales'].plot(kind='hist', bins=50, figsize=(9, 4))
plt.title('주문 금액 분포')
plt.xlabel('주문 금액')
plt.show()

## 거의 아무것도 안 보입니다

맨 왼쪽에 막대 하나가 있고 나머지는 바닥에 붙어 있습니다.
**큰 값 몇 개 때문에 가로축이 10만까지 늘어나서** 대부분이 뭉개진 것입니다.

1,000 이하만 잘라서 다시 그려 봅시다.

In [ ]:
orders[orders['Sales'] < 1000]['Sales'].plot(kind='hist', bins=50, figsize=(9, 4))
plt.title('주문 금액 분포 (1,000 미만만)')
plt.xlabel('주문 금액')
plt.show()

## 이제 모양이 보입니다

**왼쪽에 몰려 있고 오른쪽으로 길게 꼬리를 끄는 모양**입니다.
이런 걸 "오른쪽으로 치우쳤다(우편향)"고 합니다.

매출, 소득, 방문 횟수, 체류 시간 — **업무에서 만나는 숫자는 대부분 이 모양입니다.**
그리고 이 모양에서는 **평균이 중앙값보다 항상 큽니다.**

> 앞에서 평균 239, 중앙값 53.7 이 나온 이유가 이 그림에 다 있습니다.
> **분포를 한 번 그려 봤다면 평균만 보고할 일이 없습니다.**

## 상자그림 — 한눈에 요약해서 보기

히스토그램이 모양을 보여 준다면, 상자그림은 **위치와 흩어짐**을 요약해서 보여 줍니다.
여러 그룹을 나란히 비교할 때 특히 편합니다.

In [ ]:
orders[orders['Sales'] < 1000].boxplot(column='Sales', by='Segment', figsize=(9, 4))
plt.suptitle('')
plt.title('고객 유형별 주문 금액 (1,000 미만)')
plt.ylabel('주문 금액')
plt.show()

- **상자** = 가운데 50% (25% 지점 ~ 75% 지점)
- **상자 안의 선** = 중앙값
- **위아래 선(수염)** = 대략적인 범위
- **점** = 수염 밖으로 벗어난 값들

세 유형의 상자가 거의 같은 높이에 있습니다.
**고객 유형에 따른 차이가 별로 없어 보입니다.**

> 이게 "차이가 없다"는 뜻일까요, 아니면 "눈으로는 모르겠다"는 뜻일까요?
> **이 질문은 6교시에서 다시 다룹니다.**

---
# 4-3. 표준편차 — 얼마나 흩어져 있는가

평균이 "어디쯤인가"를 말한다면, **표준편차는 "얼마나 흩어져 있는가"** 를 말합니다.

같은 평균이라도 흩어짐이 다르면 완전히 다른 데이터입니다.

In [ ]:
print('주문금액   평균 {:8.1f}   표준편차 {:8.1f}'.format(
      orders['Sales'].mean(), orders['Sales'].std()))

orders['배송일수'] = (orders['Ship Date'] - orders['Order Date']).dt.days

print('배송일수   평균 {:8.2f}   표준편차 {:8.2f}'.format(
      orders['배송일수'].mean(), orders['배송일수'].std()))

## 두 숫자를 비교해 보세요

| | 평균 | 표준편차 | 표준편차 ÷ 평균 |
|---|---|---|---|
| 주문 금액 | 239.2 | **1,209.2** | **5.1배** |
| 배송 일수 | 3.97 | 1.74 | 0.4배 |

**배송 일수는 평균 4일이고, 대부분 그 근처에 모여 있습니다.**
"보통 4일 걸립니다"라고 말해도 됩니다. 예측이 가능합니다.

**주문 금액은 평균이 239 인데 표준편차가 1,209 입니다.**
평균의 5배만큼 흩어져 있습니다. "보통 239 정도입니다"라고 말할 수 없습니다.

> ### 표준편차를 왜 찍는가
> **평균을 얼마나 믿어도 되는지 알려 주기 때문입니다.**
>
> 표준편차가 작으면 평균이 전체를 잘 대표합니다.
> 표준편차가 평균보다 크면, 그 평균은 대표값 노릇을 못 합니다.
>
> 그래서 평균을 보고할 때는 **표준편차를 함께 보는 것**이 기본입니다.

## 품목별로 보면 더 분명합니다

In [ ]:
품목별 = orders.groupby('Sub-Category')['Sales'].agg(['mean', 'std', 'count']).round(1)
품목별['흩어짐배수'] = (품목별['std'] / 품목별['mean']).round(1)

품목별.sort_values('흩어짐배수', ascending=False).head(8)

맨 위를 보세요. **Fasteners 는 평균이 37인데 표준편차가 343 입니다.**
평균의 9배가 넘습니다. 클립이나 나사 같은 저가 소모품인데 흩어짐이 이렇게 클 리가 없습니다.

**뭔가 이상합니다.** 다음 절에서 확인합니다.

---
# 4-4. 이상치 — 찾는 것보다 판단이 어렵다

**이상치**는 다른 값들과 유난히 동떨어진 값입니다.
찾는 방법은 간단한데, **어떻게 할지 정하는 게 어렵습니다.**

## 찾는 방법 — IQR

가운데 50% 범위(IQR)를 기준으로, 그 범위의 1.5배를 벗어나면 이상치로 봅니다.
상자그림의 "수염 밖 점"과 같은 기준입니다.

In [ ]:
Q1 = orders['Sales'].quantile(0.25)
Q3 = orders['Sales'].quantile(0.75)
IQR = Q3 - Q1

상한 = Q3 + 1.5 * IQR

print('Q1(25%): {:8.1f}'.format(Q1))
print('Q3(75%): {:8.1f}'.format(Q3))
print('상한    : {:8.1f}'.format(상한))
print()
print('이상치: {}건 ({:.1f}%)'.format((orders['Sales'] > 상한).sum(),
                                   (orders['Sales'] > 상한).mean() * 100))

## 1,183건이 이상치로 잡혔습니다. 전체의 11.6%입니다.

여기서 그냥 지우면 **매출의 상당 부분이 사라집니다.**
앞에서 봤듯이 상위 1%가 매출의 23%를 차지하고 있었죠.

> **IQR 은 "이상하다"를 판정해 주지 않습니다.**
> 그냥 "다른 값들에 비해 크다"를 계산해 줄 뿐입니다.
> 우편향 데이터에서는 정상적인 큰 주문도 전부 여기 걸립니다.
>
> **판단은 사람이 해야 합니다.**

## 큰 값들을 직접 들여다봅시다

In [ ]:
orders.nlargest(6, 'Sales')[['Order Date', 'Sub-Category', 'Sales', 'Quantity', 'Profit']]

## 두 종류가 섞여 있습니다

| | Sub-Category | Sales | Quantity | Profit | |
|---|---|---|---|---|---|
| 1 | Tables | **104,835** | 5 | **-69.9** | ??? |
| 2 | Machines | 22,638 | 6 | -1,811 | |
| 3 | Copiers | 17,500 | 5 | +8,400 | 정상 |
| 4 | Copiers | 14,000 | 4 | +6,720 | 정상 |

**3~4번은 이상하지 않습니다.** 복사기를 몇 대 샀고, 이익도 그만큼 났습니다.
금액이 클 뿐 **말이 되는 주문**입니다.

**1번은 이상합니다.**
- 테이블 5개에 104,835 — 한 개에 2만이 넘습니다
- 그런데 이익이 **-69.9** 입니다. 2만짜리 테이블을 팔았는데 손익이 70 남짓 마이너스?

**금액만 100배로 잘못 입력된 것**으로 보입니다.
숫자 하나가 아니라 **여러 열을 함께 봐야** 알 수 있었습니다.

## 반대쪽도 봅시다 — 작은 값

In [ ]:
print('Sales 가 0 인 주문:', (orders['Sales'] == 0).sum(), '건')
print('수량이 음수인 주문:', (orders['Quantity'] < 0).sum(), '건')

orders[orders['Quantity'] < 0][['Order Date', 'Sub-Category', 'Sales', 'Quantity']]

**매출이 0원인 주문**과 **수량이 음수인 주문**이 있습니다.

- 매출 0 — 사은품일 수도, 입력 누락일 수도 있습니다
- 수량 음수 — 반품 처리이거나 입력 오류입니다

2교시에서 본 온라인 소매 데이터에서는 **음수 수량이 취소를 뜻했습니다.**
같은 모양의 값이라도 **데이터마다 뜻이 다릅니다.**

> **"이 값이 왜 이렇게 생겼는지" 알아내기 전에는 지우면 안 됩니다.**
> 오류일 수도 있고, 우리가 모르는 업무 규칙일 수도 있습니다.

## 이상치를 어떻게 할 것인가 — 네 가지 선택지

| 선택 | 언제 | 이 데이터에서 |
|---|---|---|
| **고친다** | 원래 값을 알 수 있을 때 | 104,835 -> 1,048.35 (100으로 나눔) |
| **뺀다** | 명백한 오류이고 원래 값을 모를 때 | 매출 0, 수량 음수 |
| **남긴다** | 진짜로 일어난 일일 때 | 복사기 대량 주문 |
| **따로 본다** | 그 자체가 관심사일 때 | "대형 거래만 분석" |

**어느 쪽을 골랐든 보고서에 적어야 합니다.**
2교시에서 봤듯이, 무엇을 뺐는지 안 적으면 나중에 숫자가 어긋나도 원인을 찾을 수 없습니다.

In [ ]:
# 명백한 오류만 걸러낸 표를 만들어 봅니다
정리본 = orders[(orders['Sales'] > 0) & (orders['Quantity'] > 0)].copy()

# 자릿수 오류로 보이는 건은 고쳐서 씁니다
정리본.loc[정리본['Sales'] > 50000, 'Sales'] = 정리본['Sales'] / 100

print('원본  : {:>6}행   평균 {:8.1f}   최대 {:9.0f}'.format(
      len(orders), orders['Sales'].mean(), orders['Sales'].max()))
print('정리본: {:>6}행   평균 {:8.1f}   최대 {:9.0f}'.format(
      len(정리본), 정리본['Sales'].mean(), 정리본['Sales'].max()))

> 24건을 처리했을 뿐인데 평균이 눈에 띄게 바뀝니다.
> **이상치는 개수가 적어도 평균을 크게 흔듭니다.** 중앙값은 거의 안 움직입니다.

In [ ]:
print('중앙값  원본 {:.1f}  ->  정리본 {:.1f}'.format(
      orders['Sales'].median(), 정리본['Sales'].median()))

---
# 4-5. 실습 — 판단하고 근거를 적기

## 실습 1. 이익(Profit) 을 같은 방식으로 살펴보세요

In [ ]:
print('평균  : {:8.1f}'.format(orders['Profit'].mean()))
print('중앙값: {:8.1f}'.format(orders['Profit'].median()))
print('표준편차: {:6.1f}'.format(orders['Profit'].std()))
print('최소  : {:8.1f}'.format(orders['Profit'].min()))
print('최대  : {:8.1f}'.format(orders['Profit'].max()))
print()
print('적자 주문 비율: {:.1f}%'.format((orders['Profit'] < 0).mean() * 100))

In [ ]:
orders[orders['Profit'].between(-500, 500)]['Profit'].plot(
    kind='hist', bins=60, figsize=(9, 4))
plt.title('주문별 이익 분포 (-500 ~ 500)')
plt.xlabel('이익')
plt.axvline(0, color='red', linewidth=1)
plt.show()

### 여기에 적으세요

**이 텍스트 셀을 더블클릭**해서 채우세요.

1. 이익의 평균과 중앙값 중 **어느 쪽을 보고하겠습니까? 왜?**


2. 이익 분포에서 **눈에 띄는 점** 하나


3. 적자 주문이 18% 나 됩니다. 이건 **이상치입니까, 아니면 정상입니까?**


---

## 실습 2. 이상치 하나를 골라 판단하기

아래에서 이익 손실이 가장 큰 주문 다섯 건을 봅니다.

In [ ]:
orders.nsmallest(5, 'Profit')[
    ['Order Date', 'Sub-Category', 'Sales', 'Quantity', 'Discount', 'Profit']]

### 여기에 적으세요

위 다섯 건 중 **하나를 고르고** 답하세요.

- **어느 주문입니까** (날짜, 품목)


- **입력 오류로 보입니까, 실제로 일어난 일로 보입니까?**
  (`Discount` 열을 함께 보세요)


- **그렇게 판단한 근거는 무엇입니까**


- **이 건을 빼고 분석하겠습니까, 남기고 분석하겠습니까**


---

> ### 마지막으로
> 이번 시간에 한 일은 결국 **하나입니다.**
> 요약된 숫자 하나를 보고 바로 결론을 내지 않고, **그 숫자 뒤에 뭐가 있는지 확인한 것**입니다.
>
> 평균 239 뒤에는 53.7 짜리 주문 수천 건과 10만짜리 오류 하나가 있었습니다.
> **한 줄 코드로 나온 숫자를 그대로 보고하지 않는 것** — 이게 이번 시간의 전부입니다.

---
# 정리 — 오늘 쓴 것

## 코드

| 하는 일 | 코드 |
|---|---|
| 평균 / 중앙값 | `df['열'].mean()` · `.median()` |
| 표준편차 | `df['열'].std()` |
| 사분위수 | `df['열'].quantile(0.25)` |
| 전체 요약 | `df['열'].describe()` |
| 히스토그램 | `df['열'].plot(kind='hist', bins=50)` |
| 상자그림 | `df.boxplot(column='값', by='그룹')` |
| 가장 큰 / 작은 N개 | `df.nlargest(5, '열')` · `df.nsmallest(5, '열')` |
| 한글 폰트 | `!apt-get -qq install fonts-nanum` + `plt.rc('font', family='NanumGothic')` |

## 남길 것 세 가지

1. **평균과 중앙값을 함께 본다** — 크게 다르면 그 자체가 알려야 할 정보입니다
2. **표준편차는 평균을 얼마나 믿어도 되는지 알려 준다** — 평균보다 크면 대표값 노릇을 못 합니다
3. **이상치는 찾는 것보다 판단이 어렵다** — 오류인지 진짜 사건인지는 여러 열을 함께 봐야 압니다

---

### 다음 시간

지금까지 확인한 것을 **남에게 보여 주는 방법**을 다룹니다.
그리고 이런 것을 하게 됩니다 — **같은 데이터로 정반대 인상을 주는 그래프 두 개 만들기.**